In [2]:
"""
DATA LOADING AND INITIAL EXPLORATION

Purpose:
    Load the test dataset and prepare visualization libraries for analysis.
    This notebook focuses on data preprocessing and exploratory analysis
    of a sample dataset (tested.csv).

Dataset: tested.csv
Expected to contain columns like: Age, Cabin, Sex, Embarked, Survived, etc.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('tested.csv')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (418, 12)


In [ ]:
"""
EXPLORATORY DATA ANALYSIS - Dataset Overview

Purpose:
    Display fundamental information about the dataset:
    - Shape (rows, columns)
    - Data types
    - Missing value summary
    - Memory usage

This gives a quick understanding of data structure and potential issues.
"""

print("\n=== Dataset Dimensions ===")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Dataset Information ===")
df.info()

print("\n=== Missing Values Summary ===")
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})
print(missing_summary[missing_summary['Missing_Count'] > 0])

(418, 12)
passengerid      int64
survived         int64
pclass           int64
name            object
sex             object
age            float64
sibsp            int64
parch            int64
ticket          object
fare           float64
cabin           object
embarked        object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  418 non-null    int64  
 1   survived     418 non-null    int64  
 2   pclass       418 non-null    int64  
 3   name         418 non-null    object 
 4   sex          418 non-null    object 
 5   age          418 non-null    float64
 6   sibsp        418 non-null    int64  
 7   parch        418 non-null    int64  
 8   ticket       418 non-null    object 
 9   fare         417 non-null    float64
 10  cabin        91 non-null     object 
 11  embarked     418 non-null    object 
dtypes:

In [ ]:
"""
DATA PREVIEW

Display the first few rows of the dataset to understand its structure
and content.
"""

print("\n=== First 5 Rows ===")
print(df.head())

print("\n=== Last 5 Rows ===")
print(df.tail())

print("\n=== Descriptive Statistics ===")
print(df.describe().round(2))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
"""
TASK 5: HANDLE MISSING VALUES - Numerical Variables

Purpose:
    Identify and handle missing values in numerical columns.

Strategy:
    - Fill numerical columns with median values (robust to outliers)
    - Preserves data distribution better than mean
    - Prevents introducing extreme values
"""

print("\n=== Missing Values Analysis ===")
print("Columns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Handle 'age' column - use median for numerical data
if 'age' in df.columns and df['age'].isnull().sum() > 0:
    age_median = df['age'].median()
    df['age'].fillna(age_median, inplace=True)
    print(f"\n✓ Filled Age missing values with median: {age_median:.2f}")

# Handle other numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        col_median = df[col].median()
        df[col].fillna(col_median, inplace=True)
        print(f"✓ Filled {col} missing values with median: {col_median:.2f}")

print("\nRemaining missing values:")
print(df.isnull().sum().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [ ]:
"""
TASK 6: HANDLE CATEGORICAL MISSING VALUES

Purpose:
    Handle missing values in categorical columns.

Strategy:
    - Drop rows with missing categorical values OR
    - Fill with mode (most frequent category)
    - Choice depends on percentage of missing data
"""

print("\n=== Handling Categorical Missing Values ===")

# Handle 'cabin' column
if 'cabin' in df.columns:
    cabin_missing = df['cabin'].isnull().sum()
    cabin_percentage = (cabin_missing / len(df)) * 100
    
    print(f"Cabin missing values: {cabin_missing} ({cabin_percentage:.2f}%)")
    
    # For high percentage of missing (> 50%), drop the column
    if cabin_percentage > 50:
        df = df.drop('cabin', axis=1)
        print("✓ Dropped Cabin column (>50% missing)")
    else:
        # Fill with mode if missing percentage is reasonable
        cabin_mode = df['cabin'].mode()[0] if len(df['cabin'].mode()) > 0 else 'Unknown'
        df['cabin'].fillna(cabin_mode, inplace=True)
        print(f"✓ Filled Cabin with mode: {cabin_mode}")

# Handle other categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        col_mode = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col].fillna(col_mode, inplace=True)
        print(f"✓ Filled {col} with mode: {col_mode}")

print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

np.int64(0)

In [ ]:
"""
TASK 7: REMOVE DUPLICATES AND STANDARDIZE COLUMN NAMES

Purpose:
    1. Remove duplicate records from the dataset
    2. Apply consistent naming convention (snake_case)

Benefits:
    - Eliminates redundant data
    - Improves code readability
    - Ensures consistency across analysis
"""

print("\n=== Removing Duplicates ===")
print(f"Initial rows: {len(df)}")

# Remove complete duplicates
df_duplicates_removed = df.drop_duplicates()
duplicates_count = len(df) - len(df_duplicates_removed)

df = df_duplicates_removed
print(f"Duplicates removed: {duplicates_count}")
print(f"Final rows: {len(df)}")

print("\n=== Standardizing Column Names ===")
print("Original column names:")
print(df.columns.tolist())

# Apply snake_case naming convention
df.columns = [
    col.lower()              # Convert to lowercase
       .replace(' ', '_')    # Replace spaces with underscores
       .replace('-', '_')    # Replace dashes with underscores
       .strip('_')           # Remove leading/trailing underscores
    for col in df.columns
]

print("\nStandardized column names:")
print(df.columns.tolist())
print("\n✓ Column names standardized to snake_case")

In [ ]:
"""
TASK 8: ENCODE CATEGORICAL VARIABLES

Purpose:
    Convert categorical variables to numerical format for analysis.

Methods:
    1. Label Encoding: For binary categorical variables
       - Converts categories to integers (0, 1)
       - Suitable for binary features
    
    2. One-Hot Encoding: For multi-class categorical variables
       - Creates binary columns for each category
       - Avoids ordinal relationship assumption

Example:
    - Binary (0/1): Sex → sex_encoded
    - Multi-class: Embarked → port_s, port_c, port_q
"""

from sklearn.preprocessing import LabelEncoder

print("\n=== Encoding Categorical Variables ===")

# Step 1: Label Encoding for Binary Variables
if 'sex' in df.columns:
    print("\n1. Label Encoding for 'sex' (Binary variable):")
    label_encoder = LabelEncoder()
    df['sex_encoded'] = label_encoder.fit_transform(df['sex'])
    print(f"   Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
    print(f"   ✓ Created 'sex_encoded' column")

# Step 2: One-Hot Encoding for Multi-class Variables
if 'embarked' in df.columns:
    print("\n2. One-Hot Encoding for 'embarked' (Multi-class variable):")
    df = pd.get_dummies(df, columns=['embarked'], prefix='port', drop_first=False)
    print(f"   Created columns: {[col for col in df.columns if col.startswith('port_')]}")
    print(f"   ✓ One-hot encoding complete")

# Step 3: Remove Unnecessary Columns
print("\n3. Removing Unnecessary Columns:")
columns_to_drop = ['name', 'ticket', 'cabin']
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]

if existing_cols_to_drop:
    df = df.drop(existing_cols_to_drop, axis=1)
    print(f"   Dropped columns: {existing_cols_to_drop}")
    print(f"   ✓ Removed redundant/non-numeric columns")

print("\n=== Final Prepared Dataset ===")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nColumn summary:")
print(df.dtypes)


--- Final Prepared Data ---
   passengerid  survived  pclass     sex   age  sibsp  parch     fare  \
0          892         0       3    male  34.5      0      0   7.8292   
1          893         1       3  female  47.0      1      0   7.0000   
2          894         0       2    male  62.0      0      0   9.6875   
3          895         0       3    male  27.0      0      0   8.6625   
4          896         1       3  female  22.0      1      1  12.2875   

   sex_encoded  port_C  port_Q  port_S  
0            1   False    True   False  
1            0   False   False    True  
2            1   False    True   False  
3            1   False   False    True  
4            0   False   False    True  
